In [ ]:
!pip install langchain_openai

In [ ]:
CHECKIN_MOODS = [
  "abused",
  "accepted",
  "amazed",
  "angry",
  "anxious",
  "appreciated",
  "ashamed",
  "awkward",
  "bored",
  "bullied",
  "calm",
  "confused",
  "content",
  "depressed",
  "devastated",
  "disappointed",
  "disgusted",
  "empowered",
  "entertained",
  "envious",
  "excited",
  "excluded",
  "frantic",
  "grateful",
  "guilty",
  "happy",
  "hopeful",
  "horrified",
  "insecure",
  "interested",
  "intimate",
  "jealous",
  "joyful",
  "lonely",
  "loved",
  "nervous",
  "optimistic",
  "overwhelmed",
  "peaceful",
  "powerful",
  "proud",
  "regretful",
  "relaxed",
  "resentful",
  "resigned",
  "sad",
  "safe",
  "satisfied",
  "scared",
  "self hatred",
  "shy",
  "sick",
  "smart",
  "startled",
  "stressed",
  "tired",
  "traumatized"
]

CHECKIN_SENSATIONS = [
  "pain",
  "low energy",
  "insomnia",
  "heart palpitations",
  "shortness of breath",
  "fever - body temperature",
  "weight changes",
  "constipation",
  "skin changes",
  "swelling (edema)",
  "acid reflux",
  "brain fog",
  "diarrhea",
  "pms",
  "good",
  "energized",
  "light",
  "buzzy",
  "bad",
  "jerky",
  "jumbly",
  "quivery",
  "sore",
  "achy",
  "pins and needles",
  "tickly",
  "paralyzed",
  "sharp",
  "radiating",
  "electric",
  "tense",
  "tight",
  "clenched",
  "knotted",
  "trembling",
  "quaking",
  "tingly",
  "fluttery",
  "shivery",
  "bubbly",
  "shuddering",
  "prickly",
  "blocked",
  "congested",
  "constricted",
  "bloated",
  "puffy",
  "breathless",
  "suffocating",
  "faint",
  "cool",
  "airy",
  "cold",
  "chills",
  "goose-bumps",
  "hot",
  "flushed",
  "burning",
  "floaty",
  "fluid",
  "heavy",
  "stiff",
  "thick",
  "dense",
  "empty",
  "churning",
  "sweaty",
  "damp",
  "moist",
  "flaccid",
  "clammy",
  "dull",
  "dizzy",
  "nauseous",
  "fuzzy",
  "numb",
  "weak",
  "wobbly",
  "warm",
  "vibrating",
  "itchy",
  "twitchy",
  "blood glucose - low",
  "blood glucose - high",
  "blood pressure - high",
  "blood pressure - low",
  "gas"
]

CHECKIN_BODY_PARTS = [
  "right hand",
  "right wrist",
  "right forearm",
  "right elbow",
  "right upper arm",
  "right shoulder",
  "neck",
  "left shoulder",
  "left upper arm",
  "left elbow",
  "left wrist",
  "left hand",
  "forehead",
  "head",
  "back of head",
  "eyes",
  "nose",
  "face",
  "jaw",
  "mouth",
  "chest",
  "abdomen",
  "pelvis",
  "upper back",
  "middle back",
  "lower back",
  "buttocks",
  "hip flexors",
  "right thigh",
  "right knee",
  "right calf",
  "right ankle",
  "right foot",
  "left foot",
  "left ankle",
  "left calf",
  "left knee",
  "left thigh"
]


BODY_MOOD_LABEL_EXTRACTION_TEMPLATE ="""
  You are experienced behavioral therapist capable of identifying human moods and bodily sensations that can extract the labels of body parts, body sensations and mood labels from the given input message. If there are no relevant labels, output should be an empty list".
  The input message is given in terms of either a ABC check-in or a free text message. Your goal is to output a JSON object with two keys: "body", which would have a list of key value pairs of "body_part" and if mentioned then its related "sensation" and an "intensity", "mood", which would have a list of key value pairs of "name" and its related "intensity".
  The labels extracted should be relevant labels extracted from the input message and its inferred intensity from a scale of 1 to 10 based on the language taking cues of the various sensations or moods.
  The body part and sensations need to be mapped correctly in order to establish its relationship presented in the sentence.
  If there are no relevant labels for a body_part, the value should be an array containing a single string "N/A" but if there are not a single body label or mood, return empty list.
  Here are some important rules:
  <rules>
  - If there are no labels for body or mood, output an empty list.
  - You must avoid negations or free text.
  - Parse using a case insensitive approch.
  - Focus on physical sensations and body parts from the approved list only.
  - You must avoid or skip extraction and return N/A if the input message contains violent behavior or harmful behavior.
  - You must only choose from the following predefined labels for body parts. DO NOT GENERATE ANYTHING OTHER THAN THESE PRE-DEFINED LABELS:
  ```{body_parts}```
  - You must only choose from the following predefined labels for body sensations. DO NOT GENERATE ANYTHING OTHER THAN THESE PRE-DEFINED LABELS:
  ```{sensations}```
  - You must only choose from the following predefined labels for mood. DO NOT GENERATE ANYTHING OTHER THAN THESE PRE-DEFINED LABELS:
  ```{moods}```
  </rules>

  Here are examples of various input messages and their corresponding output JSON objects
  <examples>
      1. Input Message: 'I have a headache and my stomach hurts. I feel anxious and stressed.'
        Output: {{ "body":[{{"body_part": "head", "sensation": {{"name": "pain", "intensity": 4}}}}, {{"body_part":"abdomen","sensation": {{"name":"pain", "intensity": 4}}}}], "mood": [{{"name":"anxious"}}, {{"name":"stressed"}}] }}
      2. Input Message: 'My back is sore and I feel tired.'
        Output: {{ "body":[{{"body_part": "lower back", "sensation": {{"name": "sore", "intensity": 3}}}}], "mood": [{{"name":"tired"}}] }}
      3. Input Message: 'I feel happy and excited.'
        Output: {{ "body": [], "mood": [{{"name":"happy"}}, {{"name":"excited"}}] }}
      4. Input Message: 'I feel zoned out and my legs are shaking.'
        Output: {{ "body": [{{"body_part": "right thigh", "sensation": {{"name": "trembling", "intensity": 5}}}}, {{"body_part": "left thigh", "sensation": {{"name": "trembling", "intensity": 5}}}}], "mood": [] }}
      5. Input Message: 'I have a cold and my throat is sore.'
        Output: {{ "body": [{{"body_part": "mouth", "sensation": {{"name": "sore", "intensity": 3}}}}], "mood": [] }}
      6. Input Message: 'I feel sad and lonely.'
        Output: {{ "body": [], "mood": [{{"name":"sad"}}, {{"name":"lonely"}}] }}
      7. Input Message: 'My arms not feeling numb anymore.'
        Output: {{ "body": [], "mood": [] }}
      8. Input Message: 'I was sweating alot and felt really anxious during the meeting.'
        Output: {{ "body": [{{"body_part": "N/A", "sensation": {{"name": "sweaty", "intensity": 8}}}}], "mood": [{{"name":"anxious", "intensity": 9}}] }}
      9. Input Message: 'I want to kill myself.'
        Output:  {{"body": [], "mood": [] }}
      10. Input Message: 'I had a sharp pain down my spine.'
        Output: {{ "body": [{{"body_part": "middle back", "sensation": {{"name": "pain", "intensity": 9}}}}], "mood": [] }}
      11. Input Message: 'I was tired from driving all night and my nails were hurting.'
        Output: {{ "body": [{{"body_part": "N/A", "sensation": {{"name": "pain", "intensity": 5}} }}], "mood": [{{"name":"tired", "intensity": 4}} ] }}

  </examples>
  Here is the input message -
  ```{message}```
  """



In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict
from langchain.output_parsers import PydanticOutputParser

class BodyMoodLabels(BaseModel):
    body: List[Dict] = Field(default_factory=list, description="List of body parts with sensations")
    mood: List[Dict] = Field(default_factory=list, description="List of moods")

parser = PydanticOutputParser(pydantic_object=BodyMoodLabels)

def normalize_label(label: str, approved_list: set) -> str:
    """Normalize a label by stripping emojis/special chars and checking against approved list."""
    words = re.findall(r"[A-Za-z]+", label)  # remove emojis/special chars
    if not words:
        return None
    clean_label = " ".join(words).lower()
    return clean_label if clean_label in approved_list else None

def get_body_mood_labels(body_data:dict, mood_data:dict) -> tuple[str, str]:
        """
        Filters non-approved body/mood check-ins and replaces empty labels with N/A and returns a concatenated string of Body and Mood labels
        Builds concatenate strin of body parts and sensations/moood and their intensity
        """

        body_parts = []
        for bp in body_data:
            part_name = normalize_label(bp["body_part"], CHECKIN_BODY_PARTS)
            if not part_name:
                continue
            sensations = []
            for s in bp["sensations"]:
                sens_name = normalize_label(s["name"], CHECKIN_SENSATIONS)
                if sens_name:
                    sensations.append(f"{sens_name}({s['intensity']})")
            if sensations:
                # wrap multiple sensations in {}
                if len(sensations) > 1:
                    body_parts.append(f"{part_name}: {{{', '.join(sensations)}}}")
                else:
                    body_parts.append(f"{part_name}: {sensations[0]}")
        body_str = ", ".join(body_parts) if body_parts else "N/A"

        # process mood
        mood_parts = []
        for m in mood_data:
            mood_name = normalize_label(m["name"], CHECKIN_MOODS)
            if mood_name:
                mood_parts.append(f"{mood_name}({m['intensity']})")
        mood_str = ", ".join(mood_parts) if mood_parts else "N/A"


        return body_str, mood_str

In [ ]:
import os
from langchain.chains import LLMChain
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate

_moods = ", ".join(CHECKIN_MOODS)
_body_parts = ", ".join(CHECKIN_BODY_PARTS)
_sensations = ", ".join(CHECKIN_SENSATIONS)

OPENAI_API_VERSION='API_VERSION'
OPENAI_API_BASE='API_BASE'
OPENAI_API_KEY='API_KEY'

llm = AzureChatOpenAI(
    azure_deployment="ryan-gpt",
    api_version=OPENAI_API_VERSION,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_API_BASE,
)

_body_mood_prompt = PromptTemplate(
    template=BODY_MOOD_LABEL_EXTRACTION_TEMPLATE,
    input_variables=["message"],
    partial_variables={
        "body_parts": _body_parts,
        "sensations": _sensations,
        "moods": _moods,
        "format_instructions": parser.get_format_instructions()
    }
)

_body_mood_prompt_llm_chain = _body_mood_prompt | llm



In [ ]:
def extract_body_mood_labels(message: str) -> dict:

    try:
        # predicted_labels = BotHelper.invoke_model(prompt)
        response = _body_mood_prompt_llm_chain.invoke({"message": message})
        parsed = parser.parse(response.content)
        return dict(parsed)

    except Exception as e:
        # logger.error(f"Error extracting Body/Mood labels: {e}")
        return {"body": [], "mood": []}


In [ ]:
if __name__ == '__main__':
    tests = [
        "I started sweating during the exam.",
        "My arms felt cold when I spoke.",
        "My eyes felt tingly.",
        "My legs felt numb when I stood up.",
        "My hands went numb.",
        "It was a stressful day.",
        "We were deciding on what to watch for movie night, and misty called dawn an sad bitch. Then they fought over the TV remote.",
        "They called me an angry fool.",
        "Whenever I’m driving or sitting still for a long time, feeling lonely and bored, my head gets hot, and I end up biting my nails nonstop while zoning out. When I catch myself, I stop—but later, I realize my fingers are back in my mouth again."
        "No one got hurt."
    ]
    for ex in tests:
        print(f"Example: {ex}\nResult: {extract_body_mood_labels(ex)}\n")